In [ ]:
import pandas as pd
import numpy as np 
bd = pd.read_csv(r'C:\Users\MSI\Downloads\PFA emna\bd_validation.csv')


In [ ]:
import pickle
clf_svm = pickle.load(open("model_svm.pkl", "rb"))
clf_nb = pickle.load(open("model_naive_bayes.pkl", "rb"))
clf_lr = pickle.load(open("model_logistic_regression.pkl", "rb"))

In [ ]:
labels = np.array(bd['decision'])
features = bd.drop(['decision', 'Unnamed: 0', 'm-pay_sub_msisdn', 'm-pay_sub_client_bankcode'], axis=1)
feature_list = list(features.columns)
features = np.array(features)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn import metrics


pred_svm = clf_svm.predict(features)
pred_nb = clf_nb.predict(features)
pred_lr = clf_lr.predict(features)

acc_svm = accuracy_score(labels, pred_svm)
acc_nb = accuracy_score(labels, pred_nb)
acc_lr = accuracy_score(labels, pred_lr)

print(f"Accuracy SVM : {acc_svm:.4f}")
#print(metrics.confusion_matrix(labels, pred_svm))
#print(metrics.classification_report(labels, pred_svm))
print(f"Accuracy Naive Bayes : {acc_nb:.4f}")
print(f"Accuracy Logistic Regression : {acc_lr:.4f}")
#print(metrics.confusion_matrix(labels, pred_lr))
#print(metrics.classification_report(labels, pred_lr))

In [ ]:
comparaison = pd.DataFrame({
    'Réalité (decision)': labels,
    'Prédiction SVM': pred_svm,
    'Prédiction Naive Bayes': pred_nb,
    'Prédiction Logistic Regression': pred_lr
})

print(comparaison)

In [ ]:
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

print(comparaison)

In [ ]:
!pip install shap --user

In [ ]:
stats_moyennes = pd.DataFrame(clf_nb.theta_, columns=feature_list, index=clf_nb.classes_)
stats_variances = pd.DataFrame(clf_nb.var_, columns=feature_list, index=clf_nb.classes_)

print("=== Moyennes apprises par classe ===")
print(stats_moyennes)

print("\n=== Variances apprises par classe ===")
print(stats_variances)

In [ ]:
print(feature_list)

In [ ]:
import shap
np.random.seed(42)
explainer = shap.KernelExplainer(clf_nb.predict_proba, features)

shap_values = explainer.shap_values(features)

# Vue globale : quelles features influencent le plus les décisions
shap.summary_plot(shap_values[:, :, 1], features, feature_names=feature_list)

In [ ]:
print(shap_values.shape)

In [ ]:
client_index = 15 

shap.force_plot(
    explainer.expected_value[1],
    shap_values[client_index, :, 1],
    features[client_index],
    feature_names=feature_list,
    matplotlib=True
)